# 📘 Notebook 1 — Ministral-8B-Instruct-2410
## Extraction de définitions juridiques maritimes (FR + EN)

### Pourquoi Ministral-8B ?
- **Meilleur modèle 7B francophone** : pré-entraîné sur un large corpus FR/EN, supérieur à LLaMA-2 sur les benchmarks français (Hellaswag-FR, ARC-FR)
- **Instruction-following précis** : la version Instruct v0.3 supporte le format `[INST]...[/INST]` optimisé pour les extractions structurées
- **Fenêtre contextuelle 32K** : indispensable pour les PDFs juridiques longs (MARPOL, CBD)
- **Légèreté** : tourne en 4-bit sur T4 (≈6 GB VRAM), compatible Kaggle gratuit
- **Licence Apache 2.0** : usage commercial libre


In [1]:
# ─── Installation des dépendances ───────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes pdfplumber PyMuPDF pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 106.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 66.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.0 MB/s eta 0:00:00:00:01


In [2]:
import torch
import json
import re
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import pdfplumber
import fitz  # PyMuPDF fallback
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU disponible : True
GPU : Tesla T4
VRAM : 15.6 GB


In [4]:
# ─── Configuration générale ──────────────────────────────────────────────────
MODEL_NAME = "mistralai/Ministral-8B-Instruct-2410"
OUTPUT_DIR = Path("/kaggle/working/output/mistral")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Documents à traiter (extrait du YAML de config)
DOCUMENTS = [
    {"id": "I001", "label": "Chalutage de fond", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/chalutage-fond/clalut_fond.pdf"},
    {"id": "I002", "label": "Chasse à la baleine", "lang": "en",
     "path": "/kaggle/input/datasets/assoumanasouley/convention-international/raw/raw/Baleine/ICRW_convention.pdf"},
    {"id": "I003", "label": "Construction littorale", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention-international/raw/raw/construction_sur_le_littorale/protocole_GIZC.pdf"},
    {"id": "I004", "label": "Extraction de sable", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention-international/raw/raw/extraction_Sable/convention_cbd.pdf"},
    {"id": "I005a", "label": "Rejet hydrocarbures - Protocole", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention-international/raw/raw/Rejet_dhydrocarbure/94ig4_4_protocol_fre.pdf"},
    {"id": "I005b", "label": "Rejet hydrocarbures - MARPOL Annexe I", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention-international/raw/raw/Rejet_dhydrocarbure/Annexe1consolidee_Marpol.pdf"},
    {"id": "I006", "label": "Sacs plastique - MARPOL Annexe V", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention-international/raw/raw/Sac_plastique/ANNEXEV_marpol.pdf"},
    {"id": "I007", "label": "Peintures TBT - AFS Convention", "lang": "en",
     "path": "/kaggle/input/datasets/assoumanasouley/convention-international/raw/raw/TBT/AFS_convention.pdf"},
    {"id": "I008", "label": "Oiseaux marins - CMS", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention-international/raw/raw/Oiseaux_marin/CMS_oiseau_marin.pdf"},
]

In [12]:
# ─── Extraction du texte PDF ─────────────────────────────────────────────────
def extract_pdf_text(pdf_path: str, max_chars: int = 24000) -> str:
    """
    Extrait le texte d'un PDF avec pdfplumber (meilleur pour les tableaux/colonnes).
    Fallback sur PyMuPDF si le PDF est scanné ou corrompu.
    """
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if len(text.strip()) < 100:  # PDF scanné → fallback PyMuPDF
            raise ValueError("Texte trop court, possible scan")
    except Exception as e:
        print(f"  [WARNING] pdfplumber échoué ({e}), fallback PyMuPDF...")
        try:
            doc = fitz.open(pdf_path)
            for page in doc:
                text += page.get_text() + "\n"
        except Exception as e2:
            print(f"  [ERROR] PyMuPDF aussi échoué : {e2}")
            return ""
    
    # Nettoyage basique
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text[:max_chars]  # Limite la fenêtre contextuelle


# Test sur le premier document
test_doc = DOCUMENTS[0]
if Path(test_doc["path"]).exists():
    sample = extract_pdf_text(test_doc["path"])
    print(f"Extrait ({len(sample)} chars) :\n{sample[:500]}")
else:
    print(f"[INFO] Fichier non trouvé : {test_doc['path']} (normal en dev local)")

Extrait (24000 chars) :
672
Le présent document élabore la Classification statistique internationale type des engins de
pêche (CSITEP) révisée, telle qu’approuvée et adoptée en vue de sa mise en œuvre par le
Groupe de travail chargé de coordonner les statistiques de pêche (CWP) de la FAO à
l’occasion de sa vingtcinquième session en février 2016 à Rome, en Italie. La classification
s’applique aux pêches commerciales, de subsistance et récréatives en mer et en eau douce. Le
document fournit des définitions et des illustr


In [35]:
PROMPT_FR = """Tu es un expert en droit maritime international.

Voici un EXTRAIT d'un document juridique. Ton unique tâche : extraire TOUTES les définitions présentes dans cet extrait.

Les définitions sont introduites par : signifie, désigne, s'entend de, est défini comme, aux fins de la présente, ou apparaissent dans des sections « Définitions » / « Interprétation » / « Règle 1 », ou dans des listes numérotées/lettrées sous un article définitionnel. Chaque terme peut figurer entre guillemets « », en gras, en italique, ou comme expression autonome immédiatement suivie de sa définition.

Pour chaque définition trouvée, produis un objet JSON avec ces champs EXACTS :
- "terme" : le terme défini (texte exact du document)
- "definition" : la définition littérale complète (texte exact, sans paraphrase)
- "nom_scientifique" : nom scientifique entre parenthèses si présent, sinon null
- "reference" : article, règle ou paragraphe source (ex: "Article 2", "Règle 1", "§3"), sinon null
- "exclusions" : toute exclusion ou restriction explicitement mentionnée, sinon null
- "langue" : "fr" ou "en" selon la langue du terme

RÈGLES ABSOLUES :
- Retourne UNIQUEMENT un tableau JSON valide : [ {{...}}, {{...}} ]
- Aucun texte avant ou après le tableau
- Aucun markdown, aucun backtick, aucune explication
- Si aucune définition n'est présente dans cet extrait : retourne exactement []

EXTRAIT DU DOCUMENT :
{text}"""

PROMPT_EN = """You are an expert in international maritime law.

Here is an EXTRACT from a legal document. Your only task: extract ALL definitions present in this extract.

Definitions are introduced by: means, shall mean, refers to, is defined as, for the purposes of, or appear in sections titled "Definitions" / "Interpretation", or in numbered/lettered lists under a definitional article. Each term may appear in quotes, bold, italic, or as a standalone expression immediately followed by its definition.

For each definition found, produce a JSON object with these EXACT fields:
- "terme" : the defined term (exact text from the document)
- "definition" : the complete literal definition (exact text, no paraphrase)
- "nom_scientifique" : scientific name in parentheses if present, otherwise null
- "reference" : source article, rule or paragraph (e.g. "Article 1", "Rule 2", "§3"), otherwise null
- "exclusions" : any explicitly mentioned exclusions or restrictions, otherwise null
- "langue" : "fr" or "en" depending on the language of the term

ABSOLUTE RULES :
- Return ONLY a valid JSON array: [ {{...}}, {{...}} ]
- No text before or after the array
- No markdown, no backticks, no explanation
- If no definitions are present in this extract: return exactly []

DOCUMENT EXTRACT:
{text}"""

def get_prompt(lang: str, text: str) -> str:
    template = PROMPT_FR if lang == "fr" else PROMPT_EN
    return template.format(text=text)

In [7]:
# ─── Chargement du modèle en 4-bit ──────────────────────────────────────────
print(f"Chargement de {MODEL_NAME} en 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 = meilleure qualité
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,      # Quantification double pour économiser VRAM
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("✅ Modèle chargé avec succès")
if torch.cuda.is_available():
    print(f"VRAM utilisée : {torch.cuda.memory_allocated()/1e9:.2f} GB")

Chargement de mistralai/Ministral-8B-Instruct-2410 en 4-bit...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/327 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Modèle chargé avec succès
VRAM utilisée : 1.57 GB


In [39]:
def extract_relevant_sections(text: str) -> str:
    collected = []

    # ── Stratégie 1 : sections titrées (toutes casses) ───────────────────────
    # Couvre : DEFINITIONS, Definitions, definitions, INTERPRETATION, etc.
    title_patterns = [
        r'((?:Article|ARTICLE|Rule|RULE|Règle 1|RÈGLE|Section|SECTION|Paragraph|PARAGRAPH|Para\.?)\s*[\dIVXivx]+\s*[-–.]?\s*(?:DÉFINITIONS?|Définitions?|définitions?|DEFINITIONS?|Definitions?|definitions?|INTERPRÉTATION|Interprétation|interprétation|INTERPRETATION|Interpretation|interpretation|MEANING|Meaning|meaning)\b.*?)(?=\n(?:Article|ARTICLE|Rule|RULE|Règle|RÈGLE|Section|SECTION|Paragraph|PARAGRAPH)\s*[\dIVXivx]+\b)',
        r'((?:DÉFINITIONS?|Définitions?|définitions?|DEFINITIONS?|Definitions?|definitions?|INTERPRÉTATION|Interprétation|interprétation|INTERPRETATION|Interpretation|interpretation|MEANING|Meaning|meaning)\s*\n+.*?)(?=\n{2,}[A-ZÀÂÉÈÊËÎÏÔÙÛÜ]{2}|\nArticle|\nARTICLE|\nRule|\nRULE|\nSection|\nSECTION)',
    ]
    for pattern in title_patterns:
        matches = re.findall(pattern, text, re.DOTALL)
        collected.extend(matches)

    # ── Stratégie 2 : pattern "terme" means/signifie (toutes casses) ─────────
    trigger = (
        r'(?:'
        r'[Mm][Ee][Aa][Nn][Ss]?\b'                          # means / mean / MEANS
        r'|[Ss][Hh][Aa][Ll][Ll]\s+[Mm][Ee][Aa][Nn]\b'      # shall mean / SHALL MEAN
        r'|[Rr][Ee][Ff][Ee][Rr][Ss]?\s+[Tt][Oo]\b'         # refers to / REFERS TO
        r'|[Ii][Ss]\s+[Dd][Ee][Ff][Ii][Nn][Ee][Dd]\s+[Aa][Ss]\b'  # is defined as
        r'|[Ss][Ii][Gg][Nn][Ii][Ff][Ii][Ee]\b'             # signifie / SIGNIFIE
        r'|[Dd][Éé][Ss][Ii][Gg][Nn][Ee]\b'                 # désigne / DÉSIGNE
        r'|[Ss][\""][Ee][Nn][Tt][Ee][Nn][Dd]\s+(?:[Dd][Ee]|[Cc][Oo][Mm][Mm][Ee])\b'  #entend de/comme
        r'|[Ee][Ss][Tt]\s+[Dd][Éé][Ff][Ii][Nn][Ii]\s+[Cc][Oo][Mm][Mm][Ee]\b'        # est défini comme
        r'|[Aa][Uu][Xx]\s+[Ff][Ii][Nn][Ss]\s+(?:[Dd][Ee]\s+[Ll][Aa]\s+[Pp][Rr][Éé][Ss][Ee][Nn][Tt][Ee]|[Dd][Uu]\s+[Pp][Rr][Éé][Ss][Ee][Nn][Tt])\b'  # aux fins de la présente
        r')'
    )

    # Terme entre guillemets (droits, français, allemands) ou TOUT EN MAJUSCULES
    term = (
        r'(?:'
        r'«\s*[^»]{1,80}\s*»'       # «terme»
        r'|"\s*[^"]{1,80}\s*"'      # "terme"
        r'|"\s*[^"]{1,80}\s*"'      # "terme" (guillemets typographiques)
        r'|„[^"]{1,80}"'            # „terme" (allemand)
        r"|'[^']{1,80}'"            # 'terme'
        r'|[A-ZÀÂÉÈÊËÎÏÔÙÛÜ][A-ZÀÂÉÈÊËÎÏÔÙÛÜ\s\-]{2,50}(?=\s)'  # TOUT EN MAJUSCULES
        r'|[A-ZÀÂÉÈÊËÎÏÔÙÛÜ][a-zàâéèêëîïôùûüa-z\s\-]{2,50}(?=\s)'  # Première lettre majuscule
        r')'
    )

    pattern_inline = re.compile(
        rf'({term}\s*(?:\([^)]*\))?\s*{trigger}.*?)(?=\n\s*\n|\Z|{term}\s*(?:\([^)]*\))?\s*{trigger})',
        re.DOTALL
    )
    matches2 = pattern_inline.findall(text)
    collected.extend(matches2)

    # ── Stratégie 3 : listes numérotées/lettrées ─────────────────────────────
    pattern_list = re.compile(
        rf'(?:^\s*(?:\d+\.|[a-zA-Z]\)|[ivxIVX]+\.)\s+)({term}\s*(?:\([^)]*\))?\s*{trigger}.*?)(?=\n\s*(?:\d+\.|[a-zA-Z]\)|[ivxIVX]+\.)\s|\n\s*\n|\Z)',
        re.DOTALL | re.MULTILINE
    )
    matches3 = pattern_list.findall(text)
    collected.extend(matches3)

    # ── Résultat ──────────────────────────────────────────────────────────────
    if collected:
        seen = set()
        deduped = []
        for item in collected:
            key = item.strip()[:100]
            if key not in seen:
                seen.add(key)
                deduped.append(item.strip())

        result = "\n\n".join(deduped)

        if len(result) < 200:
            print(f"  [WARNING] Sections trop courtes ({len(result)} chars), chunking document entier")
            return text

        print(f"  [INFO] {len(deduped)} bloc(s) trouvé(s) → {len(result)} chars ciblés")
        return result

    print(f"  [WARNING] Aucune section détectée, chunking document entier")
    return text

In [27]:
# ─── DIAGNOSTIC : voir la structure réelle des PDFs ──────────────────────────
for doc in DOCUMENTS[:2]:
    doc_path = Path(doc["path"])
    if not doc_path.exists():
        continue
    
    text = extract_pdf_text(str(doc_path))
    print(f"\n{'='*60}")
    print(f"[{doc['id']}] {doc['label']}")
    print(f"Taille totale : {len(text)} chars")
    print(f"\n--- DÉBUT DU TEXTE (500 chars) ---")
    print(text[:500])
    print(f"\n--- RECHERCHE MOT-CLÉ 'défin' ---")
    
    # Trouver toutes les occurrences autour de "défin"
    for m in re.finditer(r'.{100}[Dd]éfin.{100}', text):
        print(f"  pos {m.start()} → {m.group()!r}")
        print()
    
    # Même chose pour "means" / "interpretation"
    print(f"\n--- RECHERCHE MOT-CLÉ 'means' / 'interpretation' ---")
    for m in re.finditer(r'.{0,80}(?:means|interpretation|Interpretation).{0,80}', text):
        print(f"  pos {m.start()} → {m.group()!r}")
        print()


[I001] Chalutage de fond
Taille totale : 191377 chars

--- DÉBUT DU TEXTE (500 chars) ---
672
Le présent document élabore la Classification statistique internationale type des engins de
pêche (CSITEP) révisée, telle qu’approuvée et adoptée en vue de sa mise en œuvre par le
Groupe de travail chargé de coordonner les statistiques de pêche (CWP) de la FAO à
l’occasion de sa vingtcinquième session en février 2016 à Rome, en Italie. La classification
s’applique aux pêches commerciales, de subsistance et récréatives en mer et en eau douce. Le
document fournit des définitions et des illustr

--- RECHERCHE MOT-CLÉ 'défin' ---

--- RECHERCHE MOT-CLÉ 'means' / 'interpretation' ---

[I002] Chasse à la baleine
Taille totale : 58088 chars

--- DÉBUT DU TEXTE (500 chars) ---
International Convention
for the
Regulation of Whaling, 1946
Schedule
As amended by the Commission at the 69th Meeting
Lima, Peru, September 2024
1
International Convention
for the
Regulation of Whaling, 1946
Schedule
EXPLANATO

In [40]:
# ─── Pipeline principal d'extraction ────────────────────────────────────────
all_definitions = []
results_by_doc = {}

for doc in tqdm(DOCUMENTS, desc="Extraction des définitions"):
    doc_path = Path(doc["path"])
    print(f"\n{'='*60}")
    print(f"[{doc['id']}] {doc['label']} ({doc['lang'].upper()})")
    print(f"Fichier : {doc_path.name}")
    
    if not doc_path.exists():
        print(f"  [SKIP] Fichier introuvable")
        continue
    
    # Extraction du texte
    text = extract_pdf_text(str(doc_path))
    if not text:
        print(f"  [SKIP] Texte vide (PDF scanné sans OCR ?)")
        continue
    print(f"  Texte extrait : {len(text)} caractères")
    
    # Extraction des définitions
    defs = extract_definitions_mistral(text, doc["lang"], doc["id"])
    print(f"  → {len(defs)} définitions extraites")
    
    all_definitions.extend(defs)
    results_by_doc[doc["id"]] = {
        "label": doc["label"],
        "lang": doc["lang"],
        "nb_definitions": len(defs),
        "definitions": defs
    }
    
    # Sauvegarde intermédiaire
    with open(OUTPUT_DIR / f"{doc['id']}_definitions.json", "w", encoding="utf-8") as f:
        json.dump(defs, f, ensure_ascii=False, indent=2)

print(f"\n✅ Total : {len(all_definitions)} définitions extraites sur {len(DOCUMENTS)} documents")

Extraction des définitions:   0%|          | 0/9 [00:00<?, ?it/s]


[I001] Chalutage de fond (FR)
Fichier : clalut_fond.pdf
  Texte extrait : 191377 caractères
  [INFO] 1 bloc(s) trouvé(s) → 162812 chars ciblés
  [INFO] Chunking du document entier (191377 chars)
  [INFO] 36 chunk(s) à traiter
    → Chunk 1/36 (5934 chars)... 1 définitions
    → Chunk 2/36 (5996 chars)... 2 définitions
    → Chunk 3/36 (5978 chars)... 19 définitions
    → Chunk 4/36 (5996 chars)... 2 définitions
    → Chunk 5/36 (5983 chars)... 2 définitions
    → Chunk 6/36 (5978 chars)... 2 définitions
    → Chunk 7/36 (5961 chars)... 2 définitions
    → Chunk 8/36 (5930 chars)... 4 définitions
    → Chunk 9/36 (5954 chars)... 4 définitions
    → Chunk 10/36 (5957 chars)... 4 définitions
    → Chunk 11/36 (5930 chars)... 3 définitions
    → Chunk 12/36 (5916 chars)... 2 définitions
    → Chunk 13/36 (5927 chars)... 2 définitions
    → Chunk 14/36 (5992 chars)... 2 définitions
    → Chunk 15/36 (5977 chars)... 4 définitions
    → Chunk 16/36 (5982 chars)... 11 définitions
    → Chunk 

In [41]:
# ─── Sauvegarde des résultats ────────────────────────────────────────────────

# JSON complet
with open(OUTPUT_DIR / "all_definitions_mistral.json", "w", encoding="utf-8") as f:
    json.dump(all_definitions, f, ensure_ascii=False, indent=2)
print(f"JSON sauvegardé : {OUTPUT_DIR / 'all_definitions_mistral.json'}")

# CSV pour comparaison
if all_definitions:
    df = pd.DataFrame(all_definitions)
    csv_path = OUTPUT_DIR / "all_definitions_mistral.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"CSV sauvegardé : {csv_path}")
    print(f"\nAperçu des données :")
    display(df.head(10))
else:
    print("Aucune définition à sauvegarder.")

# Résumé par document
print("\n📊 Résumé par document :")
summary = []
for doc_id, res in results_by_doc.items():
    summary.append({"id": doc_id, "label": res["label"], 
                    "lang": res["lang"], "nb_definitions": res["nb_definitions"]})
df_summary = pd.DataFrame(summary)
display(df_summary)

JSON sauvegardé : /kaggle/working/output/mistral/all_definitions_mistral.json
CSV sauvegardé : /kaggle/working/output/mistral/all_definitions_mistral.csv

Aperçu des données :


,terme,definition,nom_scientifique,reference,exclusions,langue,doc_id,modele
0,Classification statistique internationale type...,Le présent document élabore la Classification ...,None,Document technique de la FAO sur les pêches et...,None,fr,I001,ministral-8b-instruct-2410
1,engins de pêche,Engins utilisés pour capturer des poissons.,None,Définitions des engins de pêche,None,fr,I001,ministral-8b-instruct-2410
2,Filets tournants,Filets utilisés pour capturer des poissons en ...,None,Définitions des engins de pêche,None,fr,I001,ministral-8b-instruct-2410
3,filets-pièges fixes non couverts,Filets-pièges fixes non couverts,None,8.1,None,fr,I001,ministral-8b-instruct-2410
4,nasses (casiers),Nasses (casiers),None,8.2,None,fr,I001,ministral-8b-instruct-2410
5,verveux,Verveux,None,8.3,None,fr,I001,ministral-8b-instruct-2410
6,filets à l'étalage (diables),Filets à l'étalage (diables),None,8.4,None,fr,I001,ministral-8b-instruct-2410
7,"barrages, parcs et bordigues","Barrages, parcs et bordigues",None,8.5,None,fr,I001,ministral-8b-instruct-2410
8,pièges aériens,Pièges aériens,None,8.6,None,fr,I001,ministral-8b-instruct-2410
9,lignes à main et lignes à cannes (manœuvrées à...,Lignes à main et lignes à cannes (manœuvrées à...,None,9.1,None,fr,I001,ministral-8b-instruct-2410



📊 Résumé par document :


,id,label,lang,nb_definitions
0,I001,Chalutage de fond,fr,113
1,I002,Chasse à la baleine,en,2
2,I003,Construction littorale,fr,7
3,I004,Extraction de sable,fr,21
4,I005a,Rejet hydrocarbures - Protocole,fr,2
5,I005b,Rejet hydrocarbures - MARPOL Annexe I,fr,89
6,I006,Sacs plastique - MARPOL Annexe V,fr,10
7,I007,Peintures TBT - AFS Convention,en,23
8,I008,Oiseaux marins - CMS,fr,14


In [42]:
# ─── Métriques de qualité ────────────────────────────────────────────────────
print("\n📈 Métriques d'extraction (Ministral-8B) :")
if all_definitions:
    df_all = pd.DataFrame(all_definitions)
    
    # Champs manquants
    for col in ['terme', 'definition', 'reference', 'nom_scientifique', 'exclusions']:
        if col in df_all.columns:
            null_pct = df_all[col].isnull().mean() * 100
            print(f"  - {col} : {null_pct:.1f}% manquants")
    
    # Longueur moyenne des définitions
    if 'definition' in df_all.columns:
        avg_len = df_all['definition'].str.len().mean()
        print(f"  - Longueur moyenne des définitions : {avg_len:.0f} chars")
    
    # Distribution par langue
    if 'langue' in df_all.columns:
        print(f"  - Distribution par langue :")
        print(df_all['langue'].value_counts().to_string())
    
    # Distribution par document
    if 'doc_id' in df_all.columns:
        print(f"  - Distribution par document :")
        print(df_all['doc_id'].value_counts().to_string())
else:
    print("  Aucune donnée à analyser.")


📈 Métriques d'extraction (Ministral-8B) :
  - terme : 0.0% manquants
  - definition : 0.0% manquants
  - reference : 17.4% manquants
  - nom_scientifique : 98.9% manquants
  - exclusions : 100.0% manquants
  - Longueur moyenne des définitions : 132 chars
  - Distribution par langue :
langue
fr    251
en     30
  - Distribution par document :
doc_id
I001     113
I005b     89
I007      23
I004      21
I008      14
I006      10
I003       7
I005a      2
I002       2
